In [30]:
import pandas as pd


path = "/Users/avivajiang/Downloads/HI-Small_Trans.csv"   

df = pd.read_csv(path, nrows=500_000, header=0,
                 names=["timestamp", "from_bank", "from_acct", "to_bank", "to_acct",
                        "amt_received", "cur_received", "amt_paid", "cur_paid",
                        "payment_format", "is_laundering"])
df["timestamp"] = pd.to_datetime(df["timestamp"], format="%Y/%m/%d %H:%M")
df = df.sort_values("timestamp").reset_index(drop=True)
print(len(df), df["is_laundering"].sum())    

500000 193


In [32]:
audit = {
    "Rows analyzed": len(df),
    "Missing values (total)": int(df.isna().sum().sum()),
    "Exact duplicate rows": int(df.duplicated().sum()),
    "Zero or negative amounts": int((df["amt_paid"] <= 0).sum()),
    "Self-transfers (sender = receiver)": int((df["from_acct"] == df["to_acct"]).sum()),
    "Rows where currencies differ": int((df["cur_received"] != df["cur_paid"]).sum()),
    "Distinct currencies": df["cur_paid"].nunique(),
}
audit_df = pd.DataFrame(audit.items(), columns=["Check", "Result"])
print(audit_df)
print(df["payment_format"].value_counts())

                                Check  Result
0                       Rows analyzed  500000
1              Missing values (total)       0
2                Exact duplicate rows       0
3            Zero or negative amounts       0
4  Self-transfers (sender = receiver)  297563
5        Rows where currencies differ    3145
6                 Distinct currencies      15
payment_format
Reinvestment    285394
Cheque           81362
Credit Card      59045
ACH              30531
Cash             21250
Bitcoin          13914
Wire              8504
Name: count, dtype: int64


In [34]:
tx = df[df["from_acct"] != df["to_acct"]].copy()          

tx["sender_recipients"] = tx.groupby("from_acct")["to_acct"].transform("nunique")

tx["flag_ach"]    = tx["payment_format"] == "ACH"
tx["flag_fanout"] = tx["sender_recipients"] >= 10
tx["flag_large"]  = tx["amt_paid"] >= 10_000
tx["flag_count"]  = tx[["flag_ach", "flag_fanout", "flag_large"]].sum(axis=1)

In [44]:
total_launder = int(tx["is_laundering"].sum())

def score(mask, name):
    alerts = int(mask.sum())                                  
    hits = int((mask & (tx["is_laundering"] == 1)).sum())    
    return {"scenario": name, "alerts": alerts, "true_hits": hits,
            "false_alarms": alerts - hits,
            "precision": round(hits / alerts, 4) if alerts else 0,
            "recall": round(hits / total_launder, 3),
            "pct_of_txns_flagged": round(alerts / len(tx) * 100, 2)}

results = pd.DataFrame([
    score(tx.flag_ach, "ACH only"),
    score(tx.flag_fanout, "Fan-out only"),
    score(tx.flag_large, "Large amount only"),
    score(tx.flag_ach & tx.flag_fanout, "ACH + fan-out"),
    score(tx.flag_ach & tx.flag_large, "ACH + large"),
    score(tx.flag_fanout & tx.flag_large, "Fan-out + large"),
    score(tx.flag_count >= 2, "Any 2 of 3 flags"),
    score(tx.flag_count == 3, "All 3 flags"),
])
print(results.to_string())


            scenario  alerts  true_hits  false_alarms  precision  recall  pct_of_txns_flagged
0           ACH only   27436        164         27272     0.0060   0.854                13.55
1       Fan-out only   18348         56         18292     0.0031   0.292                 9.06
2  Large amount only   72371        100         72271     0.0014   0.521                35.75
3      ACH + fan-out      28         28             0     1.0000   0.146                 0.01
4        ACH + large   11476         88         11388     0.0077   0.458                 5.67
5    Fan-out + large    7311         27          7284     0.0037   0.141                 3.61
6   Any 2 of 3 flags   18785        113         18672     0.0060   0.589                 9.28
7        All 3 flags      15         15             0     1.0000   0.078                 0.01


In [40]:
with pd.ExcelWriter("AML_Analysis.xlsx") as xw:
    audit_df.to_excel(xw, sheet_name="Data_Audit", index=False)
    results.to_excel(xw, sheet_name="Scenarios", index=False)
    tuning.to_excel(xw, sheet_name="Tuning", index=False)
print("saved")

saved


In [42]:
import os; print(os.getcwd())

/Users/avivajiang
